# 🌿 Algerian Daridja Sentiment Analysis
**Backend:** data loading → exploration → preprocessing → Bi-LSTM → evaluation → saving  ·  **Frontend:** Gradio web app

Run the cells **from top to bottom** (`Runtime → Change runtime type → T4 GPU` is recommended).
Two datasets = two label systems, so **one model per dataset**: DarijaDZ_SA (3 classes, MIX removed) and Zenodo (2 classes).

In [ ]:
# ============ 1. SETUP: libraries, seeds, colour palette ============
!pip -q install datasets openpyxl wordcloud arabic-reshaper python-bidi "gradio>=5,<6"

import os, re, json, glob, random, unicodedata, warnings, html, requests
from collections import Counter
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, matplotlib as mpl, seaborn as sns
import tensorflow as tf
from tensorflow.keras import layers, regularizers
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.utils.class_weight import compute_class_weight
warnings.filterwarnings('ignore')

SEED = 42
random.seed(SEED); np.random.seed(SEED); tf.keras.utils.set_random_seed(SEED)

# ---- Colour palette ----
FIG_LEAF, SAGE, CREAM = '#526B45', '#A8B58A', '#F4E9D5'
ROSE, BURGUNDY, AUBERGINE = '#C98D86', '#823B4D', '#492638'
CLASS_COLOR = {'NEG': BURGUNDY, 'NEU': ROSE, 'POS': FIG_LEAF}
CLASS_NAME  = {'NEG': 'Negative', 'NEU': 'Neutral', 'POS': 'Positive'}
sns.set_theme(style='whitegrid')
plt.rcParams.update({'figure.facecolor': CREAM, 'axes.facecolor': '#FBF6EC', 'axes.edgecolor': AUBERGINE,
                     'axes.labelcolor': AUBERGINE, 'text.color': AUBERGINE, 'xtick.color': AUBERGINE,
                     'ytick.color': AUBERGINE, 'axes.titleweight': 'bold', 'grid.color': SAGE, 'grid.alpha': .35})

# ---- Folders (saved inside Colab's temporary storage; download /content/daridja_sentiment to keep it) ----
ROOT = '/content/daridja_sentiment'
os.makedirs(ROOT, exist_ok=True)
CLASSES = {'darijadz': ['NEG', 'NEU', 'POS'],   # DarijaDZ_SA (3 classes)
           'zenodo':   ['NEG', 'POS']}          # Zenodo      (2 classes)
TITLES  = {'darijadz': 'DarijaDZ_SA (3 classes)', 'zenodo': 'Zenodo (2 classes)'}
print('TensorFlow', tf.__version__, '| GPU:', tf.config.list_physical_devices('GPU'))

In [ ]:
# ============ 2. LOAD BOTH DATASETS + BASIC STATISTICS ============
from datasets import load_dataset

# ---- Dataset 1: DarijaDZ_SA (Hugging Face) -> drop the MIX class ----
hf = load_dataset('nasrellahkharroubi/DarijaDZ_SA')
dz = pd.concat([hf[s].to_pandas() for s in hf.keys()], ignore_index=True)[['text', 'label']]
dz['label'] = dz['label'].astype(str).str.strip().str.upper()
print('DarijaDZ_SA before removing MIX:\n', dz['label'].value_counts().to_string())
dz = dz[dz['label'].isin(['NEG', 'NEU', 'POS'])].reset_index(drop=True)      # MIX removed
assert 'MIX' not in set(dz['label'])

# ---- Dataset 2: Zenodo (each Excel cell is one line "label,comment": 0 = negative, 1 = positive) ----
ZEN_PATH = f'{ROOT}/zenodo.xlsx'
ZEN_URL = 'https://zenodo.org/records/10937412/files/Sentiment_dataset_of_Algerian_dialect%20%2811760%29.xlsx?download=1'
if not os.path.exists(ZEN_PATH):
    try:
        r = requests.get(ZEN_URL, timeout=90); r.raise_for_status()
        open(ZEN_PATH, 'wb').write(r.content)
    except Exception as e:                                   # fallback: manual upload
        print('Download failed:', e, '-> please upload the .xlsx from https://zenodo.org/records/10937412')
        from google.colab import files
        os.replace(list(files.upload().keys())[0], ZEN_PATH)

cells_ = pd.read_excel(ZEN_PATH, header=None, dtype=str)
lines = cells_.iloc[:, 0] if cells_.shape[1] == 1 else cells_.fillna('').agg(','.join, axis=1)
LINE_RE = re.compile(r'^\s*"?\s*(0|1|0\.0|1\.0)\s*"?\s*,\s*(.*)$', re.S)
rows = []
for line in lines.dropna().astype(str):
    m = LINE_RE.match(line)
    if m:                                                    # the header line "label,text" is skipped
        rows.append({'text': m.group(2).strip().strip('"').strip(),
                     'label': 'NEG' if m.group(1).startswith('0') else 'POS'})
zen = pd.DataFrame(rows)

# ---- Basic statistics + class distribution ----
RAW = {'darijadz': dz, 'zenodo': zen}
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, (key, df) in zip(axes, RAW.items()):
    cnt = df['label'].value_counts().reindex(CLASSES[key])
    print(f'\n=== {TITLES[key]} === samples: {len(df)} | duplicates: {df["text"].duplicated().sum()} '
          f'| avg words/comment: {df["text"].str.split().str.len().mean():.1f}')
    print(pd.DataFrame({'count': cnt, 'percent': (cnt / cnt.sum() * 100).round(1)}).to_string())
    bars = ax.bar([CLASS_NAME[c] for c in cnt.index], cnt.values, color=[CLASS_COLOR[c] for c in cnt.index])
    for b, v in zip(bars, cnt.values):
        ax.text(b.get_x() + b.get_width() / 2, v, f'{v}\n({v / cnt.sum() * 100:.1f}%)', ha='center', va='bottom')
    ax.set_title(TITLES[key] + ' - class distribution'); ax.set_ylim(0, cnt.max() * 1.2)
plt.tight_layout(); plt.show()

In [ ]:
# ============ 3. PREPROCESSING (cleaning + tokenization) AND DATASET CLEANING ============
_URL   = re.compile(r'(https?://\S+|www\.\S+|\S+@\S+\.\S+)', re.I)
_MENT  = re.compile(r'\[MENTION\]|@\w+', re.I)
_EMOJI = re.compile('[\U0001F000-\U0001FAFF\U00002600-\U000027BF\U00002B00-\U00002BFF\U0000FE00-\U0000FE0F'
                    '\U0000200B-\U0000200F\U000020E3]+')
_AR = {'أ':'ا','إ':'ا','آ':'ا','ٱ':'ا','ى':'ي','ی':'ي','ئ':'ي','ؤ':'و','ة':'ه',
       'ک':'ك','گ':'ك','ڤ':'ف','ڨ':'ق','پ':'ب','چ':'ج'}                      # unify Arabic / Daridja letters
for i, d in enumerate('٠١٢٣٤٥٦٧٨٩'): _AR[d] = str(i)                       # Arabic digits -> 0-9
for i, d in enumerate('۰۱۲۳۴۵۶۷۸۹'): _AR[d] = str(i)
_AR = str.maketrans(_AR)
_KEEP, _REPEAT = re.compile(r'[^\u0621-\u064Aa-z0-9\s]'), re.compile(r'(.)\1{2,}')

def preprocess_text(text):
    """Raw comment -> cleaned string. Digits are KEPT (Arabizi: 3, 7, 9...) and negations are NOT removed."""
    if text is None: return ''
    t = _URL.sub(' ', str(text)); t = _MENT.sub(' ', t); t = _EMOJI.sub(' ', t)   # URLs, mentions, emojis
    t = t.translate(_AR).replace('\u0640', '')                                   # unify letters, remove tatweel
    t = ''.join(c for c in unicodedata.normalize('NFKD', t) if not unicodedata.combining(c)).lower()  # diacritics
    t = _KEEP.sub(' ', t)                                                        # punctuation / symbols -> space
    t = _REPEAT.sub(r'\1\1', t)                                                  # bezzzzaf -> bezzaf
    return ' '.join(tok for tok in t.split() if not tok.isdigit())               # tokenize, drop pure numbers

for e in ["Merci bcp 3azizti lbere7 jarabtou ja raw3a 💖 https://youtu.be/abc", "[MENTION] ماشي مليح هاد الشي!!! خسااااارة"]:
    print('BEFORE:', e, '\nAFTER :', preprocess_text(e), '\n')

# ---- Apply to both datasets: drop empties, conflicting labels and duplicates ----
CLEAN = {}
for key, df in RAW.items():
    df = df.dropna().copy(); n0 = len(df)
    df['text_raw'] = df['text'].astype(str)
    df['text_clean'] = df['text_raw'].map(preprocess_text)
    df = df[df['text_clean'] != '']                                              # nothing left after cleaning
    df = df[df.groupby('text_clean')['label'].transform('nunique') == 1]         # same text, different labels
    CLEAN[key] = df.drop_duplicates('text_clean')[['text_raw', 'text_clean', 'label']].reset_index(drop=True)
    print(f'{TITLES[key]}: {n0} -> {len(CLEAN[key])} samples after cleaning')
# the same comment must not exist in both datasets
overlap = set(CLEAN['darijadz']['text_clean']) & set(CLEAN['zenodo']['text_clean'])
CLEAN['zenodo'] = CLEAN['zenodo'][~CLEAN['zenodo']['text_clean'].isin(overlap)].reset_index(drop=True)

In [ ]:
# ============ 4. WORD CLOUDS: BEFORE (top row) vs AFTER (bottom row) PREPROCESSING ============
import arabic_reshaper
from bidi.algorithm import get_display
from wordcloud import WordCloud

FONT = os.path.join(mpl.get_data_path(), 'fonts/ttf/DejaVuSans.ttf')           # has Arabic glyphs
for pat in ['/usr/share/fonts/**/NotoNaskhArabic*Regular*.ttf', '/usr/share/fonts/**/Amiri*Regular*.ttf']:
    if glob.glob(pat, recursive=True): FONT = glob.glob(pat, recursive=True)[0]; break
WC_COLORS = [FIG_LEAF, BURGUNDY, AUBERGINE, ROSE]
color_fn = lambda *a, **k: random.choice(WC_COLORS)

for key, df in CLEAN.items():
    classes = CLASSES[key]
    fig, axes = plt.subplots(2, len(classes), figsize=(6 * len(classes), 9), squeeze=False)
    for r, (col, stage) in enumerate([('text_raw', 'BEFORE'), ('text_clean', 'AFTER')]):
        for c, cls in enumerate(classes):
            freq = Counter()
            for t in df.loc[df['label'] == cls, col]: freq.update(t.split())
            top = {get_display(arabic_reshaper.reshape(w)): n for w, n in freq.most_common(300) if len(w) > 1}
            wc = WordCloud(width=700, height=450, background_color=CREAM, font_path=FONT, max_words=80,
                           color_func=color_fn, random_state=SEED).generate_from_frequencies(top)
            axes[r, c].imshow(wc, interpolation='bilinear'); axes[r, c].axis('off')
            axes[r, c].set_title(f'{CLASS_NAME[cls]} - {stage} preprocessing')
    fig.suptitle(f'{TITLES[key]} - word clouds', fontsize=16, fontweight='bold')
    plt.tight_layout(); plt.show()

In [ ]:
# ============ 5. TRAIN / VALIDATION / TEST SPLIT (70/15/15) + TOKENIZATION ============
DATA = {}
for key, df in CLEAN.items():
    classes = CLASSES[key]; label2id = {c: i for i, c in enumerate(classes)}
    train, temp = train_test_split(df, test_size=0.30, stratify=df['label'], random_state=SEED)
    val, test   = train_test_split(temp, test_size=0.50, stratify=temp['label'], random_state=SEED)

    # max length + vocabulary are learned from the TRAIN set only (no leakage)
    max_len = int(np.clip(np.percentile(train['text_clean'].str.split().str.len(), 95), 20, 150))
    vec = tf.keras.layers.TextVectorization(max_tokens=30000 if key == 'darijadz' else 15000, standardize=None,
                                            split='whitespace', output_mode='int', output_sequence_length=max_len)
    vec.adapt(tf.data.Dataset.from_tensor_slices(train['text_clean'].values).batch(512))

    enc = lambda d: vec(tf.constant(list(d['text_clean']))).numpy().astype('int32')
    DATA[key] = {'classes': classes, 'label2id': label2id, 'max_len': max_len, 'vec': vec,
                 'X_train': enc(train), 'y_train': train['label'].map(label2id).values,
                 'X_val':   enc(val),   'y_val':   val['label'].map(label2id).values,
                 'X_test':  enc(test),  'y_test':  test['label'].map(label2id).values}
    print(f'{TITLES[key]}: train={len(train)} | val={len(val)} | test={len(test)} | '
          f'max_len={max_len} | vocab={len(vec.get_vocabulary())}')

In [ ]:
# ============ 6. MODEL: Embedding + Bidirectional LSTM, TRAINING ============
def build_model(vocab_size, max_len, n_classes):
    inp = tf.keras.Input(shape=(max_len,), dtype='int32')
    x = layers.Embedding(vocab_size, 128, mask_zero=True)(inp)
    x = layers.Dropout(0.3)(x)
    x = layers.Bidirectional(layers.LSTM(64))(x)                                   # reads left->right and right->left
    x = layers.Dropout(0.4)(x)
    x = layers.Dense(64, activation='relu', kernel_regularizer=regularizers.l2(1e-4))(x)
    x = layers.Dropout(0.3)(x)
    out = layers.Dense(n_classes, activation='softmax')(x)                          # one probability per class
    m = tf.keras.Model(inp, out)
    m.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return m

MODELS, HISTORY = {}, {}
for key, d in DATA.items():
    print(f'\n########## Training {TITLES[key]} ##########')
    tf.keras.utils.set_random_seed(SEED)
    model = build_model(len(d['vec'].get_vocabulary()), d['max_len'], len(d['classes']))
    cw = compute_class_weight('balanced', classes=np.arange(len(d['classes'])), y=d['y_train'])   # class imbalance
    hist = model.fit(d['X_train'], d['y_train'], validation_data=(d['X_val'], d['y_val']),
                     epochs=20, batch_size=64, class_weight=dict(enumerate(cw)), verbose=2,
                     callbacks=[tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True),
                                tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=1)])
    MODELS[key], HISTORY[key] = model, hist.history

In [ ]:
# ============ 7. EVALUATION ON TEST SET: report, confusion matrix, learning curves ============
METRICS = {}
for key, d in DATA.items():
    model, h, classes = MODELS[key], HISTORY[key], d['classes']
    names = [CLASS_NAME[c] for c in classes]
    y_pred = model.predict(d['X_test'], batch_size=256, verbose=0).argmax(axis=1)
    test_loss, test_acc = model.evaluate(d['X_test'], d['y_test'], verbose=0)
    METRICS[key] = {'test_loss': float(test_loss), 'test_accuracy': float(test_acc)}

    print('=' * 70, f'\n{TITLES[key]} - TEST accuracy: {test_acc:.4f} | loss: {test_loss:.4f}\n' + '=' * 70)
    print(classification_report(d['y_test'], y_pred, target_names=names, digits=4))

    fig, axes = plt.subplots(1, 3, figsize=(18, 4.8))
    ep = range(1, len(h['loss']) + 1)
    axes[0].plot(ep, h['accuracy'], 'o-', color=FIG_LEAF, label='Train'); axes[0].plot(ep, h['val_accuracy'], 's-', color=BURGUNDY, label='Validation')
    axes[0].set(title='Learning curve - accuracy', xlabel='Epoch', ylabel='Accuracy'); axes[0].legend()
    axes[1].plot(ep, h['loss'], 'o-', color=FIG_LEAF, label='Train'); axes[1].plot(ep, h['val_loss'], 's-', color=BURGUNDY, label='Validation')
    axes[1].set(title='Learning curve - loss', xlabel='Epoch', ylabel='Loss'); axes[1].legend()
    sns.heatmap(confusion_matrix(d['y_test'], y_pred), annot=True, fmt='d', cmap=sns.light_palette(BURGUNDY, as_cmap=True),
                xticklabels=names, yticklabels=names, cbar=False, linewidths=1, linecolor=CREAM, ax=axes[2])
    axes[2].set(title='Confusion matrix (test set)', xlabel='Predicted', ylabel='True')
    fig.suptitle(TITLES[key], fontsize=15, fontweight='bold'); plt.tight_layout(); plt.show()

In [ ]:
# ============ 8. SAVE THE MODELS (model + vocabulary + config) AND CHECK THEY RELOAD ============
for key, d in DATA.items():
    out = f'{ROOT}/models/{key}'; os.makedirs(out, exist_ok=True)
    MODELS[key].save(f'{out}/model.keras')                                          # trained network
    vocab = d['vec'].get_vocabulary()
    json.dump(vocab[2:], open(f'{out}/vocabulary.json', 'w', encoding='utf-8'), ensure_ascii=False)   # tokenizer
    json.dump({'classes': d['classes'], 'max_len': d['max_len'], 'title': TITLES[key], **METRICS[key]},
              open(f'{out}/config.json', 'w', encoding='utf-8'), ensure_ascii=False, indent=2)

    # sanity check: reload from disk -> same predictions as the in-memory model
    reloaded = tf.keras.models.load_model(f'{out}/model.keras')
    same = np.allclose(reloaded.predict(d['X_test'][:200], verbose=0), MODELS[key].predict(d['X_test'][:200], verbose=0), atol=1e-5)
    print(f'[{key}] saved to {out} -> {sorted(os.listdir(out))} | reload check passed: {same}')

In [ ]:
# ============ 9. GUI (Gradio): loads the SAVED models from disk, full-screen, fig palette ============
import gradio as gr

def load_bundle(key):
    """Loads model + vocabulary + config from disk (nothing is retrained)."""
    p = f'{ROOT}/models/{key}'
    cfg = json.load(open(f'{p}/config.json', encoding='utf-8'))
    vec = tf.keras.layers.TextVectorization(standardize=None, split='whitespace', output_mode='int',
            output_sequence_length=cfg['max_len'], vocabulary=json.load(open(f'{p}/vocabulary.json', encoding='utf-8')))
    return {'cfg': cfg, 'vec': vec, 'model': tf.keras.models.load_model(f'{p}/model.keras')}

BUNDLES = {k: load_bundle(k) for k in ['darijadz', 'zenodo']}
CHOICES = [(b['cfg']['title'], k) for k, b in BUNDLES.items()]
EMOJI = {'NEG': '😞', 'NEU': '😐', 'POS': '😊'}

CSS = f"""
:root, .dark {{ --body-text-color:{AUBERGINE}; --body-background-fill:{CREAM}; --background-fill-primary:#FBF6EC;
  --background-fill-secondary:{CREAM}; --block-background-fill:#FBF6EC; --block-label-text-color:{AUBERGINE};
  --block-title-text-color:{AUBERGINE}; --border-color-primary:{SAGE}; --input-background-fill:#FFFBF2;
  --body-text-color-subdued:{FIG_LEAF}; --color-accent:{BURGUNDY}; --block-border-color:{SAGE}; }}
html, body {{ background:{CREAM} !important; }}
.gradio-container {{ max-width:100% !important; width:100% !important; min-height:100vh; margin:0 !important;
  padding:0 2.5vw 30px 2.5vw !important; background:{CREAM} !important; font-family:'Segoe UI',system-ui,sans-serif !important; }}
footer {{ display:none !important; }}
#hero {{ background:linear-gradient(120deg,{AUBERGINE} 0%,{BURGUNDY} 100%); color:{CREAM}; text-align:center;
  padding:34px 20px; border-radius:0 0 26px 26px; margin:0 -2.5vw 22px -2.5vw; }}
#hero h1 {{ margin:0; font-size:2.4rem; color:{CREAM}; }}
#hero p {{ max-width:900px; margin:10px auto 0 auto; line-height:1.55; color:{SAGE}; font-size:1.05rem; }}
#comment-box textarea {{ background:#FFFBF2 !important; color:{AUBERGINE} !important; border:2px solid {SAGE} !important;
  border-radius:14px !important; font-size:1.1rem !important; }}
#predict-btn {{ background:{BURGUNDY} !important; color:{CREAM} !important; border:none !important; border-radius:14px !important;
  font-weight:700 !important; font-size:1.1rem !important; padding:14px !important; }}
#predict-btn:hover {{ background:{AUBERGINE} !important; }}
.card {{ background:#FBF6EC; border:2px solid {SAGE}; border-radius:20px; padding:24px 28px; color:{AUBERGINE}; min-height:340px; }}
.badge {{ display:inline-block; background:{SAGE}; color:{AUBERGINE}; border-radius:999px; padding:5px 14px; font-size:.85rem; font-weight:600; }}
.result-label {{ font-size:2.2rem; font-weight:800; margin:14px 0 2px 0; }}
.bar-row {{ margin:14px 0; }}
.bar-head {{ display:flex; justify-content:space-between; font-weight:600; margin-bottom:5px; }}
.bar-bg {{ background:{CREAM}; border:1px solid {SAGE}; border-radius:999px; height:18px; overflow:hidden; }}
.bar-fill {{ height:100%; border-radius:999px; }}
.small {{ color:{FIG_LEAF}; font-size:.9rem; margin-top:16px; word-break:break-word; }}
.warn {{ border-color:{ROSE}; }}
"""

HEADER = """<div id="hero"><h1>🌿 Algerian Daridja Sentiment Analysis</h1>
<p>Type a comment in Algerian Daridja (Arabic script, Arabizi or mixed with French), choose the classification system,
and the Bidirectional LSTM will predict its sentiment with a confidence score for every class.</p></div>"""
PLACEHOLDER = '<div class="card"><span class="badge">Result</span><p style="color:%s;margin-top:14px">Your prediction will appear here ✨</p></div>' % FIG_LEAF

def predict_ui(text, key):
    if not text or not text.strip():
        return '<div class="card warn"><b>⚠ Please type a comment first.</b></div>'
    b = BUNDLES[key]; classes = b['cfg']['classes']
    clean = preprocess_text(text)                                                   # same cleaning as in training
    if not clean:
        return '<div class="card warn"><b>⚠ Nothing left to analyse after cleaning (only emojis / links / symbols).</b></div>'
    probs = b['model'](b['vec'](tf.constant([clean])).numpy().astype('int32'), training=False).numpy()[0]
    best = classes[int(np.argmax(probs))]
    bars = ''.join(f'<div class="bar-row"><div class="bar-head"><span>{EMOJI[c]} {CLASS_NAME[c]}</span><span>{p*100:.1f}%</span></div>'
                   f'<div class="bar-bg"><div class="bar-fill" style="width:{p*100:.1f}%;background:{CLASS_COLOR[c]}"></div></div></div>'
                   for c, p in zip(classes, probs))
    return f"""<div class="card"><span class="badge">{html.escape(b['cfg']['title'])}</span>
      <div class="result-label" style="color:{CLASS_COLOR[best]}">{EMOJI[best]} {CLASS_NAME[best]}</div>
      <div style="color:{FIG_LEAF}">Model confidence: {probs.max()*100:.1f}%</div>
      <hr style="border:none;border-top:1px dashed {SAGE};margin:16px 0"><b>Confidence for every class</b>{bars}
      <div class="small"><b>Text seen by the model:</b><br>{html.escape(clean)}</div></div>"""

try: demo.close()
except Exception: pass

with gr.Blocks(css=CSS, title='Algerian Daridja Sentiment Analysis', fill_width=True) as demo:
    gr.HTML(HEADER)
    with gr.Row(equal_height=False):
        with gr.Column(scale=1):
            dataset = gr.Radio(choices=CHOICES, value=CHOICES[0][1], label='Dataset / classification system')
            comment = gr.Textbox(label='Daridja comment', lines=7, elem_id='comment-box',
                                 placeholder='اكتب تعليقك هنا ... / Ekteb commentaire taa3ek hna ...')
            btn = gr.Button('Predict Sentiment', elem_id='predict-btn')
            gr.Examples(examples=[['ربي يحفظك يا خويا بصح خدمة رائعة'], ['ماشي مليح هاد الشي، خسارة على الوقت'],
                                  ["merci bezaf 3la had l'video, bravo"], ['makach wach tdir, ga3 ykdbou 3lina']],
                        inputs=[comment], label='Try an example')
        with gr.Column(scale=1):
            output = gr.HTML(PLACEHOLDER)
    btn.click(predict_ui, inputs=[comment, dataset], outputs=output)

# inline=False -> opens the app as a FULL-PAGE public link (click the gradio.live link below) instead of a small iframe
demo.launch(share=True, inline=False, debug=True)